<a href="https://colab.research.google.com/github/ota0425/brain-tumor-adversarial-experiment/blob/main/brain_tumor_adversarial_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MRI脳腫瘍画像のAdversarial Attack検知

既存のMobileNetV2脳腫瘍4クラス分類モデルを利用し、入力MRI画像がcleanかadversarialかを判定する二値検知モデルを構築する。

このNotebookは、攻撃生成・脆弱性評価を行う`brain_tumor_adversarial_examples.ipynb`と分離する。

## 実験フェーズ

1. **実験基盤の準備**（実装済み）
2. **検知用clean/adversarialデータの生成**（実装済み）
3. **MobileNetV2特徴を使った二値検知器の学習**（実装済み）
4. **MobileNetV2上位層のfine-tuning**（実装済み・実行待ち）
5. 既知εと未知εの評価
6. 発展：PGDなど未知攻撃の評価


## Step 1. 実験基盤の準備

このステップでは、分類モデルとデータセットを再現し、検知実験の前提が整っていることを確認する。

完了条件：

- TensorFlowとGPUを認識できる
- Google Drive上のデータセットと保存済みモデルを読み込める
- Training、Validation、Testingのクラス順が一致する
- Testing 1,600枚に対するclean Accuracyが既存結果の83.19%前後になる


In [3]:
import json
import os
import random
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.20

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Random seed:", SEED)


TensorFlow version: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Random seed: 42


### 1.1 Google Driveとパスの準備

検知実験でも、攻撃実験と同じデータセットZIPと保存済みベースラインモデルを使用する。


In [4]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [5]:
DRIVE_ROOT = Path("/content/drive/MyDrive/ThammasatResearch")
DATASET_ZIP_PATH = DRIVE_ROOT / "dataset" / "archive.zip"
EXTRACT_ROOT = Path("/content/brain_tumor")
TRAINING_DIRECTORY = EXTRACT_ROOT / "Training"
TESTING_DIRECTORY = EXTRACT_ROOT / "Testing"
CLASSIFIER_MODEL_PATH = (
    DRIVE_ROOT / "models" / "baseline_mobilenetv2.keras"
)
DETECTION_RESULTS_DIRECTORY = DRIVE_ROOT / "results" / "detection"
DETECTION_MODELS_DIRECTORY = DRIVE_ROOT / "models" / "detection"

assert DATASET_ZIP_PATH.exists(), (
    f"Dataset ZIP was not found: {DATASET_ZIP_PATH}"
)
assert CLASSIFIER_MODEL_PATH.exists(), (
    f"Classifier model was not found: {CLASSIFIER_MODEL_PATH}"
)

DETECTION_RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
DETECTION_MODELS_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Dataset ZIP:", DATASET_ZIP_PATH)
print("Classifier model:", CLASSIFIER_MODEL_PATH)
print("Detection results:", DETECTION_RESULTS_DIRECTORY)
print("Detection models:", DETECTION_MODELS_DIRECTORY)


Dataset ZIP: /content/drive/MyDrive/ThammasatResearch/dataset/archive.zip
Classifier model: /content/drive/MyDrive/ThammasatResearch/models/baseline_mobilenetv2.keras
Detection results: /content/drive/MyDrive/ThammasatResearch/results/detection
Detection models: /content/drive/MyDrive/ThammasatResearch/models/detection


In [6]:
if TRAINING_DIRECTORY.exists() and TESTING_DIRECTORY.exists():
    print("Dataset is already extracted:", EXTRACT_ROOT)
else:
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP_PATH, "r") as zip_file:
        zip_file.extractall(EXTRACT_ROOT)
    print("Dataset extracted:", EXTRACT_ROOT)

assert TRAINING_DIRECTORY.exists(), TRAINING_DIRECTORY
assert TESTING_DIRECTORY.exists(), TESTING_DIRECTORY


Dataset extracted: /content/brain_tumor


### 1.2 データセットの再構築

既存分類実験と同じ画像サイズ、バッチサイズ、乱数シード、Training/Validation分割を使用する。TrainingとValidationは`subset="both"`による1回の呼び出しで同時に作成し、異なるshuffle条件によるsplitの不一致を防ぐ。さらに元画像pathの重複が0件であることを検証する。`class_names`はprefetch前に保存する。


In [ ]:
train_dataset, validation_dataset = (
    tf.keras.utils.image_dataset_from_directory(
        TRAINING_DIRECTORY,
        validation_split=VALIDATION_SPLIT,
        subset="both",
        seed=SEED,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="int",
        shuffle=True,
    )
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TESTING_DIRECTORY,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

class_names = list(train_dataset.class_names)
validation_class_names = list(validation_dataset.class_names)
test_class_names = list(test_dataset.class_names)

assert class_names == validation_class_names == test_class_names, (
    class_names,
    validation_class_names,
    test_class_names,
)

train_file_paths = set(train_dataset.file_paths)
validation_file_paths = set(validation_dataset.file_paths)
split_overlap = train_file_paths & validation_file_paths

assert not split_overlap, (
    f"Training/Validation overlap: {len(split_overlap)} files"
)
assert len(train_file_paths) == 4480, len(train_file_paths)
assert len(validation_file_paths) == 1120, (
    len(validation_file_paths)
)
assert len(train_file_paths | validation_file_paths) == 5600

print("Class names:", class_names)
print("Training source images:", len(train_file_paths))
print("Validation source images:", len(validation_file_paths))
print("Training/Validation overlap:", len(split_overlap))


In [ ]:
autotune = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(autotune)
validation_dataset = validation_dataset.prefetch(autotune)
test_dataset = test_dataset.prefetch(autotune)

sample_images, sample_labels = next(iter(test_dataset))

print("Image batch shape:", sample_images.shape)
print("Label batch shape:", sample_labels.shape)
print("Pixel range:", float(tf.reduce_min(sample_images)),
      "to", float(tf.reduce_max(sample_images)))


### 1.3 保存済み分類モデルの再現確認

検知用adversarial画像はこの分類モデルに対して生成する。検知データを作る前に、cleanのTesting結果が既存実験と一致することを確認する。


In [ ]:
classifier_model = tf.keras.models.load_model(CLASSIFIER_MODEL_PATH)
classifier_model.trainable = False

print("Classifier loaded successfully.")
print("Input shape:", classifier_model.input_shape)
print("Output shape:", classifier_model.output_shape)


In [ ]:
clean_test_loss, clean_test_accuracy = classifier_model.evaluate(
    test_dataset,
    verbose=1,
)

EXPECTED_CLEAN_ACCURACY = 0.8319
ACCURACY_TOLERANCE = 0.005

print(f"Clean Test Loss: {clean_test_loss:.4f}")
print(f"Clean Test Accuracy: {clean_test_accuracy:.4f}")
print(f"Clean Test Accuracy: {clean_test_accuracy * 100:.2f}%")

if abs(clean_test_accuracy - EXPECTED_CLEAN_ACCURACY) > ACCURACY_TOLERANCE:
    raise RuntimeError(
        "Clean accuracy does not match the recorded baseline. "
        f"Expected about {EXPECTED_CLEAN_ACCURACY:.4f}, "
        f"but received {clean_test_accuracy:.4f}."
    )

print("Step 1 completed: experiment prerequisites are reproducible.")


## Step 1 checkpoint

Step 1はここまでとする。上のセルで`Step 1 completed`が表示されたことを確認した後、次のステップで検知用clean/adversarialデータを生成する。

予定しているε（0–255スケール）：

- 検知器の学習：0.01, 0.1, 0.5
- 未知εの評価：0.05, 0.25, 1


## Step 2. clean/adversarial検知データの生成

既存分類モデルに対してuntargeted white-box FGSMを適用し、検知器の入力と二値ラベルをバッチ単位で生成する。

| 入力 | 検知ラベル |
|---|---:|
| clean MRI | 0 |
| FGSM adversarial MRI | 1 |

すべてのadversarial画像をラベル1とする。分類攻撃の成功・失敗は、後の評価でサブグループとして分ける。

大量の画像ファイルを複製せず、`tf.data`上でバッチごとに生成する。分類モデルは常に`training=False`で使用し、重みを更新しない。


In [ ]:
TRAIN_EPSILONS = (0.01, 0.1, 0.5)
UNSEEN_EPSILONS = (0.05, 0.25, 1.0)
ALL_EVALUATION_EPSILONS = (
    *TRAIN_EPSILONS,
    *UNSEEN_EPSILONS,
)

train_epsilon_tensor = tf.constant(TRAIN_EPSILONS, tf.float32)
classification_loss = (
    tf.keras.losses.SparseCategoricalCrossentropy()
)

print("Training epsilons:", TRAIN_EPSILONS)
print("Unseen evaluation epsilons:", UNSEEN_EPSILONS)


In [ ]:
@tf.function
def compute_fgsm_direction(images, tumor_labels):
    """Return clean predictions and the sign of d(loss)/d(images)."""
    images = tf.cast(images, tf.float32)
    tumor_labels = tf.cast(tumor_labels, tf.int32)

    with tf.GradientTape() as tape:
        tape.watch(images)
        clean_predictions = classifier_model(
            images,
            training=False,
        )
        loss = classification_loss(
            tumor_labels,
            clean_predictions,
        )

    gradients = tape.gradient(loss, images)
    if gradients is None:
        raise RuntimeError("FGSM input gradients could not be computed.")

    return clean_predictions, tf.sign(gradients)


@tf.function
def create_adversarial_images(images, signed_gradients, epsilon):
    """Create FGSM images on the model's 0-255 input scale."""
    images = tf.cast(images, tf.float32)
    epsilon = tf.cast(epsilon, tf.float32)
    adversarial_images = images + epsilon * signed_gradients
    return tf.clip_by_value(adversarial_images, 0.0, 255.0)


@tf.function
def build_detection_batch(
    images,
    tumor_labels,
    epsilon,
    shuffle_output=False,
):
    """Build a balanced batch of clean=0 and adversarial=1 images."""
    _, signed_gradients = compute_fgsm_direction(
        images,
        tumor_labels,
    )
    adversarial_images = create_adversarial_images(
        images,
        signed_gradients,
        epsilon,
    )

    batch_size = tf.shape(images)[0]
    detection_images = tf.concat(
        [tf.cast(images, tf.float32), adversarial_images],
        axis=0,
    )
    detection_labels = tf.concat(
        [
            tf.zeros((batch_size, 1), tf.float32),
            tf.ones((batch_size, 1), tf.float32),
        ],
        axis=0,
    )

    if shuffle_output:
        indices = tf.random.shuffle(
            tf.range(tf.shape(detection_images)[0]),
            seed=SEED,
        )
        detection_images = tf.gather(detection_images, indices)
        detection_labels = tf.gather(detection_labels, indices)

    return detection_images, detection_labels


print("FGSM detection-data functions are ready.")


### 2.1 学習パイプライン

各Trainingバッチにε=0.01, 0.1, 0.5を順番に割り当てる。各元画像に対しcleanとadversarialを1枚ずつ作り、ラベル数を1:1にする。Trainingは毎epochシャッフルされるため、画像とεの組み合わせもepoch間で変化する。


In [ ]:
def make_training_detection_batch(batch_index, source_batch):
    images, tumor_labels = source_batch
    epsilon_index = tf.cast(
        batch_index % len(TRAIN_EPSILONS),
        tf.int32,
    )
    epsilon = tf.gather(train_epsilon_tensor, epsilon_index)
    return build_detection_batch(
        images,
        tumor_labels,
        epsilon,
        shuffle_output=True,
    )


train_detection_dataset = (
    train_dataset
    .enumerate()
    .map(
        make_training_detection_batch,
        num_parallel_calls=1,
    )
    .prefetch(1)
)

print("Training detection pipeline is ready.")


### 2.2 ValidationとTestingの評価パイプライン

Validationは検知器の学習と閾値決定に使用する。Testingは最終評価専用とし、検知閾値の調整に使用しない。

Testingはεごとに独立したデータセットを作り、既知εと未知εを分けて報告できるようにする。


In [ ]:
def make_fixed_epsilon_detection_dataset(source_dataset, epsilon):
    epsilon_tensor = tf.constant(epsilon, tf.float32)

    def map_batch(images, tumor_labels):
        return build_detection_batch(
            images,
            tumor_labels,
            epsilon_tensor,
            shuffle_output=False,
        )

    return source_dataset.map(
        map_batch,
        num_parallel_calls=1,
    ).prefetch(1)


known_validation_datasets = {
    epsilon: make_fixed_epsilon_detection_dataset(
        validation_dataset,
        epsilon,
    )
    for epsilon in TRAIN_EPSILONS
}

validation_detection_dataset = None
for epsilon in TRAIN_EPSILONS:
    epsilon_dataset = known_validation_datasets[epsilon]
    if validation_detection_dataset is None:
        validation_detection_dataset = epsilon_dataset
    else:
        validation_detection_dataset = (
            validation_detection_dataset.concatenate(epsilon_dataset)
        )

known_test_detection_datasets = {
    epsilon: make_fixed_epsilon_detection_dataset(
        test_dataset,
        epsilon,
    )
    for epsilon in TRAIN_EPSILONS
}

unseen_test_detection_datasets = {
    epsilon: make_fixed_epsilon_detection_dataset(
        test_dataset,
        epsilon,
    )
    for epsilon in UNSEEN_EPSILONS
}

print("Known validation epsilons:",
      tuple(known_validation_datasets.keys()))
print("Known test epsilons:",
      tuple(known_test_detection_datasets.keys()))
print("Unseen test epsilons:",
      tuple(unseen_test_detection_datasets.keys()))


### 2.3 生成データの健全性確認

ε=0.1の1バッチを使い、画素範囲、検知ラベル数、摂動の最大値、元画像とadversarial画像を確認する。


In [ ]:
SANITY_CHECK_EPSILON = 0.1
sanity_clean_images, sanity_tumor_labels = next(iter(train_dataset))
sanity_clean_predictions, sanity_signed_gradients = (
    compute_fgsm_direction(
        sanity_clean_images,
        sanity_tumor_labels,
    )
)
sanity_adversarial_images = create_adversarial_images(
    sanity_clean_images,
    sanity_signed_gradients,
    SANITY_CHECK_EPSILON,
)
sanity_adversarial_predictions = classifier_model(
    sanity_adversarial_images,
    training=False,
)

sanity_detection_images, sanity_detection_labels = (
    build_detection_batch(
        sanity_clean_images,
        sanity_tumor_labels,
        SANITY_CHECK_EPSILON,
        shuffle_output=False,
    )
)

perturbation = (
    sanity_adversarial_images - tf.cast(sanity_clean_images, tf.float32)
)
maximum_absolute_perturbation = float(
    tf.reduce_max(tf.abs(perturbation))
)
clean_label_count = int(tf.reduce_sum(
    tf.cast(sanity_detection_labels == 0, tf.int32)
))
adversarial_label_count = int(tf.reduce_sum(
    tf.cast(sanity_detection_labels == 1, tf.int32)
))

clean_predicted_classes = tf.argmax(
    sanity_clean_predictions,
    axis=1,
)
adversarial_predicted_classes = tf.argmax(
    sanity_adversarial_predictions,
    axis=1,
)
changed_predictions = int(tf.reduce_sum(tf.cast(
    clean_predicted_classes != adversarial_predicted_classes,
    tf.int32,
)))

assert sanity_detection_images.shape[0] == 2 * sanity_clean_images.shape[0]
assert clean_label_count == adversarial_label_count
assert float(tf.reduce_min(sanity_detection_images)) >= 0.0
assert float(tf.reduce_max(sanity_detection_images)) <= 255.0
assert maximum_absolute_perturbation <= SANITY_CHECK_EPSILON + 1e-5

print("Detection batch shape:", sanity_detection_images.shape)
print("Clean labels:", clean_label_count)
print("Adversarial labels:", adversarial_label_count)
print("Maximum |perturbation|:", maximum_absolute_perturbation)
print("Changed predictions in this batch:", changed_predictions)
print("Step 2 completed: detection-data pipelines are ready.")


In [ ]:
number_of_examples = 3
fig, axes = plt.subplots(
    number_of_examples,
    3,
    figsize=(12, 4 * number_of_examples),
)

for index in range(number_of_examples):
    original = sanity_clean_images[index].numpy()
    adversarial = sanity_adversarial_images[index].numpy()
    difference = adversarial - original
    difference_display = (
        difference + SANITY_CHECK_EPSILON
    ) / (2 * SANITY_CHECK_EPSILON)

    true_name = class_names[int(sanity_tumor_labels[index])]
    clean_name = class_names[int(clean_predicted_classes[index])]
    adversarial_name = class_names[
        int(adversarial_predicted_classes[index])
    ]

    axes[index, 0].imshow(original.astype(np.uint8))
    axes[index, 0].set_title(
        f"Clean\nTrue: {true_name}, Pred: {clean_name}"
    )
    axes[index, 0].axis("off")

    axes[index, 1].imshow(
        np.clip(difference_display, 0, 1)
    )
    axes[index, 1].set_title(
        f"Perturbation\nε = {SANITY_CHECK_EPSILON}"
    )
    axes[index, 1].axis("off")

    axes[index, 2].imshow(adversarial.astype(np.uint8))
    axes[index, 2].set_title(
        f"Adversarial\nPred: {adversarial_name}"
    )
    axes[index, 2].axis("off")

plt.tight_layout()
plt.show()


## Step 2 checkpoint

Step 2はここまでとする。`Step 2 completed`が表示され、clean/adversarialが同数で、最大摂動がε以下であることを確認する。

次のStep 3では、既存MobileNetV2の内部特徴を入力とする二値検知ヘッドを作成し、検知器の学習を行う。


## Step 3. MobileNetV2特徴を使った二値検知器の学習

既存分類モデルの`GlobalAveragePooling2D`出力を内部特徴として使用し、clean/adversarialを判定する小さな検知ヘッドを接続する。

分類モデルと特徴抽出器は凍結し、検知ヘッドだけを学習する。これにより、攻撃実験で確定した腫瘍分類モデルの重みを変更しない。

検知モデルの初期構成：

~~~text
224×224×3 MRI
  → 凍結済みMobileNetV2特徴（1280次元）
  → Dense(128, ReLU, L2)
  → Dropout(0.3)
  → Dense(1, Sigmoid)
~~~


In [ ]:
feature_layer = next(
    (
        layer
        for layer in reversed(classifier_model.layers)
        if isinstance(
            layer,
            tf.keras.layers.GlobalAveragePooling2D,
        )
    ),
    None,
)

if feature_layer is None:
    raise RuntimeError(
        "GlobalAveragePooling2D layer was not found in the classifier."
    )

feature_extractor = tf.keras.Model(
    inputs=classifier_model.input,
    outputs=feature_layer.output,
    name="frozen_classifier_feature_extractor",
)
feature_extractor.trainable = False

sample_features = feature_extractor(
    sanity_clean_images,
    training=False,
)

print("Feature layer:", feature_layer.name)
print("Feature shape:", sample_features.shape)
print("Feature extractor trainable:", feature_extractor.trainable)


In [ ]:
detector_inputs = tf.keras.Input(
    shape=(*IMAGE_SIZE, 3),
    name="mri_image",
)
detector_features = feature_extractor(
    detector_inputs,
    training=False,
)
detector_hidden = tf.keras.layers.Dense(
    128,
    activation="relu",
    kernel_regularizer=tf.keras.regularizers.l2(1e-4),
    name="detector_dense",
)(detector_features)
detector_hidden = tf.keras.layers.Dropout(
    0.3,
    name="detector_dropout",
)(detector_hidden)
detector_outputs = tf.keras.layers.Dense(
    1,
    activation="sigmoid",
    name="adversarial_probability",
)(detector_hidden)

detector_model = tf.keras.Model(
    inputs=detector_inputs,
    outputs=detector_outputs,
    name="fgsm_feature_detector",
)

detector_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="binary_accuracy",
        ),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="roc_auc", curve="ROC"),
        tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
    ],
)

detector_model.summary()

trainable_parameters = int(np.sum([
    np.prod(variable.shape)
    for variable in detector_model.trainable_weights
]))
total_parameters = detector_model.count_params()

print("Trainable detector parameters:", trainable_parameters)
print("Total parameters including frozen extractor:",
      total_parameters)


### 3.1 学習条件

- Loss: Binary Crossentropy
- Optimizer: Adam, learning rate = 0.001
- 最大epoch: 20
- ベストモデルの基準: Validation ROC-AUC
- EarlyStopping patience: 4
- ReduceLROnPlateau patience: 2

Validationは学習用の既知ε=0.01, 0.1, 0.5のみを使用する。Testingと未知εは学習やモデル選択に使用しない。


In [ ]:
BASELINE_RUN_NAME = (
    "frozen_mobilenetv2_disjoint_split_baseline"
)
MAX_DETECTOR_EPOCHS = 20
DETECTOR_MODEL_PATH = (
    DETECTION_MODELS_DIRECTORY
    / f"{BASELINE_RUN_NAME}.keras"
)
DETECTOR_HISTORY_PATH = (
    DETECTION_RESULTS_DIRECTORY
    / f"{BASELINE_RUN_NAME}_history.csv"
)
DETECTOR_CURVES_PATH = (
    DETECTION_RESULTS_DIRECTORY
    / f"{BASELINE_RUN_NAME}_curves.png"
)
BASELINE_CONFIG_PATH = (
    DETECTION_RESULTS_DIRECTORY
    / f"{BASELINE_RUN_NAME}_config.json"
)
BASELINE_VALIDATION_METRICS_PATH = (
    DETECTION_RESULTS_DIRECTORY
    / f"{BASELINE_RUN_NAME}_validation_metrics.json"
)
FORCE_RETRAIN_FROZEN_BASELINE = False
RUN_FROZEN_BASELINE = (
    FORCE_RETRAIN_FROZEN_BASELINE
    or not DETECTOR_MODEL_PATH.exists()
)

baseline_config = {
    "run_name": BASELINE_RUN_NAME,
    "random_seed": SEED,
    "image_size": list(IMAGE_SIZE),
    "batch_size": BATCH_SIZE,
    "validation_split": VALIDATION_SPLIT,
    "training_source_images": len(train_file_paths),
    "validation_source_images": len(validation_file_paths),
    "training_validation_overlap": len(split_overlap),
    "training_epsilons": list(TRAIN_EPSILONS),
    "feature_layer": feature_layer.name,
    "feature_extractor_trainable": (
        feature_extractor.trainable
    ),
    "optimizer": "Adam",
    "initial_learning_rate": 1e-3,
    "maximum_epochs": MAX_DETECTOR_EPOCHS,
    "model_selection_metric": "val_roc_auc",
    "tensorflow_version": tf.__version__,
    "force_retrain": FORCE_RETRAIN_FROZEN_BASELINE,
}
BASELINE_CONFIG_PATH.write_text(
    json.dumps(baseline_config, indent=2),
    encoding="utf-8",
)

detector_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=DETECTOR_MODEL_PATH,
        monitor="val_roc_auc",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_roc_auc",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_roc_auc",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

print("Baseline run:", BASELINE_RUN_NAME)
print("Train frozen baseline:", RUN_FROZEN_BASELINE)
print("Best detector model path:", DETECTOR_MODEL_PATH)
print("Baseline config:", BASELINE_CONFIG_PATH)


In [ ]:
if RUN_FROZEN_BASELINE:
    detector_history = detector_model.fit(
        train_detection_dataset,
        validation_data=validation_detection_dataset,
        epochs=MAX_DETECTOR_EPOCHS,
        callbacks=detector_callbacks,
        verbose=1,
    )
else:
    detector_history = None
    if not DETECTOR_HISTORY_PATH.exists():
        raise FileNotFoundError(
            "Baseline model exists, but its history CSV is missing: "
            f"{DETECTOR_HISTORY_PATH}"
        )
    print("Skipped frozen-baseline training.")
    print("Using saved model:", DETECTOR_MODEL_PATH)


In [ ]:
if RUN_FROZEN_BASELINE:
    detector_history_df = pd.DataFrame(
        detector_history.history
    )
    detector_history_df.index = np.arange(
        1,
        len(detector_history_df) + 1,
    )
    detector_history_df.index.name = "epoch"
    detector_history_df.to_csv(DETECTOR_HISTORY_PATH)
else:
    detector_history_df = pd.read_csv(
        DETECTOR_HISTORY_PATH,
        index_col="epoch",
    )

metric_pairs = [
    ("loss", "Loss"),
    ("binary_accuracy", "Binary Accuracy"),
    ("roc_auc", "ROC-AUC"),
    ("pr_auc", "PR-AUC"),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for axis, (metric_name, title) in zip(axes.flat, metric_pairs):
    axis.plot(
        detector_history_df.index,
        detector_history_df[metric_name],
        label="Training",
    )
    axis.plot(
        detector_history_df.index,
        detector_history_df[f"val_{metric_name}"],
        label="Validation",
    )
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.grid(True)
    axis.legend()

plt.tight_layout()
plt.savefig(
    DETECTOR_CURVES_PATH,
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print("Saved:", DETECTOR_HISTORY_PATH)
print("Saved:", DETECTOR_CURVES_PATH)


### 3.2 ベストモデルのValidation確認

Validation全体に対する検知指標を確認する。ここでは基本閾値0.5を使う。FPR=1%の閾値決定とε別評価は後のStepで行う。


In [ ]:
best_detector_model = tf.keras.models.load_model(
    DETECTOR_MODEL_PATH
)

validation_metrics = best_detector_model.evaluate(
    validation_detection_dataset,
    return_dict=True,
    verbose=1,
)

print("Validation metrics at threshold 0.5:")
for metric_name, metric_value in validation_metrics.items():
    print(f"  {metric_name}: {metric_value:.4f}")

validation_label_batches = []
validation_score_batches = []
for validation_images, validation_labels in (
    validation_detection_dataset
):
    validation_scores = best_detector_model(
        validation_images,
        training=False,
    ).numpy().reshape(-1)
    validation_label_batches.append(
        validation_labels.numpy().reshape(-1)
    )
    validation_score_batches.append(validation_scores)

validation_labels_all = np.concatenate(
    validation_label_batches
)
validation_scores_all = np.concatenate(
    validation_score_batches
)
validation_predictions = (
    validation_scores_all >= 0.5
).astype(np.int32)

true_negatives = int(np.sum(
    (validation_labels_all == 0)
    & (validation_predictions == 0)
))
false_positives = int(np.sum(
    (validation_labels_all == 0)
    & (validation_predictions == 1)
))
false_positive_rate = (
    false_positives
    / (false_positives + true_negatives)
)

baseline_validation_metrics = {
    metric_name: float(metric_value)
    for metric_name, metric_value in validation_metrics.items()
}
baseline_validation_metrics.update({
    "false_positive_rate": float(false_positive_rate),
    "threshold": 0.5,
    "number_of_validation_examples": int(
        validation_labels_all.size
    ),
})
BASELINE_VALIDATION_METRICS_PATH.write_text(
    json.dumps(baseline_validation_metrics, indent=2),
    encoding="utf-8",
)

assert np.all(validation_scores_all >= 0.0)
assert np.all(validation_scores_all <= 1.0)

print(f"  false_positive_rate: {false_positive_rate:.4f}")
print("Adversarial-score range:",
      float(validation_scores_all.min()),
      "to",
      float(validation_scores_all.max()))
print("Saved:", BASELINE_VALIDATION_METRICS_PATH)
print("Frozen baseline completed.")


## Stop point after Step 3

Step 3はここまでとする。`Frozen baseline completed`が表示され、Google Driveにベスト検知モデル、実験条件JSON、Validation指標JSON、学習履歴CSV、学習曲線が保存されたことを確認する。これは修正済みsplitを用いたfine-tuning比較用baselineである。

次のStep 3Bでは、保存済みbaselineを初期値としてMobileNetV2上位層をfine-tuningする。


## Step 3B. MobileNetV2上位層のfine-tuning

保存済みの凍結baseline検知モデルから開始し、MobileNetV2の上位30層だけを学習可能にする。Batch Normalization層は凍結を維持し、learning rateを`1e-5`へ下げる。

baselineとfine-tuningモデルは別名で保存する。各モデルは保存済みファイルが存在すれば学習をスキップするため、`すべてのセルを実行`しても不要な再学習は行われない。再学習が必要な場合だけ`FORCE_RETRAIN_*`を`True`にする。


In [ ]:
FINE_TUNING_RUN_NAME = (
    "finetuned_mobilenetv2_top30_detector"
)
FINE_TUNING_LEARNING_RATE = 1e-5
FINE_TUNING_TOP_LAYERS = 30
MAX_FINE_TUNING_EPOCHS = 20
FORCE_RETRAIN_FINE_TUNING = False

FINE_TUNED_MODEL_PATH = (
    DETECTION_MODELS_DIRECTORY
    / f"{FINE_TUNING_RUN_NAME}.keras"
)
FINE_TUNING_HISTORY_PATH = (
    DETECTION_RESULTS_DIRECTORY
    / f"{FINE_TUNING_RUN_NAME}_history.csv"
)
FINE_TUNING_CURVES_PATH = (
    DETECTION_RESULTS_DIRECTORY
    / f"{FINE_TUNING_RUN_NAME}_curves.png"
)
FINE_TUNING_CONFIG_PATH = (
    DETECTION_RESULTS_DIRECTORY
    / f"{FINE_TUNING_RUN_NAME}_config.json"
)
FINE_TUNING_VALIDATION_METRICS_PATH = (
    DETECTION_RESULTS_DIRECTORY
    / f"{FINE_TUNING_RUN_NAME}_validation_metrics.json"
)
BASELINE_COMPARISON_PATH = (
    DETECTION_RESULTS_DIRECTORY
    / "frozen_vs_finetuned_validation.csv"
)
RUN_FINE_TUNING = (
    FORCE_RETRAIN_FINE_TUNING
    or not FINE_TUNED_MODEL_PATH.exists()
)

fine_tuned_detector_model = tf.keras.models.load_model(
    DETECTOR_MODEL_PATH
)
fine_tune_feature_extractor = (
    fine_tuned_detector_model.get_layer(
        "frozen_classifier_feature_extractor"
    )
)
mobilenet_backbone = next(
    (
        layer
        for layer in fine_tune_feature_extractor.layers
        if isinstance(layer, tf.keras.Model)
        and "mobilenetv2" in layer.name.lower()
    ),
    None,
)
if mobilenet_backbone is None:
    raise RuntimeError(
        "MobileNetV2 backbone was not found in the baseline."
    )

fine_tune_feature_extractor.trainable = True
for layer in fine_tune_feature_extractor.layers:
    layer.trainable = False

mobilenet_backbone.trainable = True
for layer in mobilenet_backbone.layers:
    layer.trainable = False
for layer in mobilenet_backbone.layers[
    -FINE_TUNING_TOP_LAYERS:
]:
    if not isinstance(
        layer,
        tf.keras.layers.BatchNormalization,
    ):
        layer.trainable = True

fine_tuned_detector_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=FINE_TUNING_LEARNING_RATE
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="binary_accuracy",
        ),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(
            name="roc_auc", curve="ROC"
        ),
        tf.keras.metrics.AUC(
            name="pr_auc", curve="PR"
        ),
    ],
)

fine_tuned_layer_names = [
    layer.name
    for layer in mobilenet_backbone.layers
    if layer.trainable
]
assert fine_tuned_layer_names, (
    "No MobileNetV2 layers were unfrozen."
)
assert all(
    not isinstance(layer, tf.keras.layers.BatchNormalization)
    for layer in mobilenet_backbone.layers
    if layer.trainable
)

fine_tuning_config = {
    "run_name": FINE_TUNING_RUN_NAME,
    "initial_model": str(DETECTOR_MODEL_PATH),
    "random_seed": SEED,
    "training_epsilons": list(TRAIN_EPSILONS),
    "requested_top_layers": FINE_TUNING_TOP_LAYERS,
    "trainable_mobilenet_layers": fine_tuned_layer_names,
    "batch_normalization_frozen": True,
    "learning_rate": FINE_TUNING_LEARNING_RATE,
    "maximum_epochs": MAX_FINE_TUNING_EPOCHS,
    "model_selection_metric": "val_roc_auc",
    "tensorflow_version": tf.__version__,
    "force_retrain": FORCE_RETRAIN_FINE_TUNING,
}
FINE_TUNING_CONFIG_PATH.write_text(
    json.dumps(fine_tuning_config, indent=2),
    encoding="utf-8",
)

print("MobileNetV2 backbone:", mobilenet_backbone.name)
print("Trainable MobileNetV2 layers:",
      len(fine_tuned_layer_names))
print("Run fine-tuning:", RUN_FINE_TUNING)
print("Fine-tuned model path:", FINE_TUNED_MODEL_PATH)


In [ ]:
fine_tuning_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=FINE_TUNED_MODEL_PATH,
        monitor="val_roc_auc",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_roc_auc",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_roc_auc",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
]

if RUN_FINE_TUNING:
    fine_tuning_history = fine_tuned_detector_model.fit(
        train_detection_dataset,
        validation_data=validation_detection_dataset,
        epochs=MAX_FINE_TUNING_EPOCHS,
        callbacks=fine_tuning_callbacks,
        verbose=1,
    )
else:
    fine_tuning_history = None
    if not FINE_TUNING_HISTORY_PATH.exists():
        raise FileNotFoundError(
            "Fine-tuned model exists, but its history is missing: "
            f"{FINE_TUNING_HISTORY_PATH}"
        )
    print("Skipped fine-tuning; using the saved model.")


In [ ]:
if RUN_FINE_TUNING:
    fine_tuning_history_df = pd.DataFrame(
        fine_tuning_history.history
    )
    fine_tuning_history_df.index = np.arange(
        1,
        len(fine_tuning_history_df) + 1,
    )
    fine_tuning_history_df.index.name = "epoch"
    fine_tuning_history_df.to_csv(
        FINE_TUNING_HISTORY_PATH
    )
else:
    fine_tuning_history_df = pd.read_csv(
        FINE_TUNING_HISTORY_PATH,
        index_col="epoch",
    )

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for axis, (metric_name, title) in zip(
    axes.flat, metric_pairs
):
    axis.plot(
        fine_tuning_history_df.index,
        fine_tuning_history_df[metric_name],
        label="Training",
    )
    axis.plot(
        fine_tuning_history_df.index,
        fine_tuning_history_df[f"val_{metric_name}"],
        label="Validation",
    )
    axis.set_title(f"Fine-tuning {title}")
    axis.set_xlabel("Epoch")
    axis.grid(True)
    axis.legend()

plt.tight_layout()
plt.savefig(
    FINE_TUNING_CURVES_PATH,
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print("Saved:", FINE_TUNING_HISTORY_PATH)
print("Saved:", FINE_TUNING_CURVES_PATH)


### 3B.1 fine-tuningモデルのValidation評価

保存されたベストモデルを閾値0.5で評価し、凍結baselineと同じ指標を比較する。Testingデータはモデル選択に使用しない。


In [ ]:
best_fine_tuned_model = tf.keras.models.load_model(
    FINE_TUNED_MODEL_PATH
)
fine_tuned_validation_metrics = (
    best_fine_tuned_model.evaluate(
        validation_detection_dataset,
        return_dict=True,
        verbose=1,
    )
)

fine_tuned_label_batches = []
fine_tuned_score_batches = []
for validation_images, validation_labels in (
    validation_detection_dataset
):
    scores = best_fine_tuned_model(
        validation_images,
        training=False,
    ).numpy().reshape(-1)
    fine_tuned_label_batches.append(
        validation_labels.numpy().reshape(-1)
    )
    fine_tuned_score_batches.append(scores)

fine_tuned_labels_all = np.concatenate(
    fine_tuned_label_batches
)
fine_tuned_scores_all = np.concatenate(
    fine_tuned_score_batches
)
fine_tuned_predictions = (
    fine_tuned_scores_all >= 0.5
).astype(np.int32)
fine_tuned_true_negatives = int(np.sum(
    (fine_tuned_labels_all == 0)
    & (fine_tuned_predictions == 0)
))
fine_tuned_false_positives = int(np.sum(
    (fine_tuned_labels_all == 0)
    & (fine_tuned_predictions == 1)
))
fine_tuned_fpr = (
    fine_tuned_false_positives
    / (
        fine_tuned_false_positives
        + fine_tuned_true_negatives
    )
)

fine_tuned_validation_results = {
    metric_name: float(metric_value)
    for metric_name, metric_value
    in fine_tuned_validation_metrics.items()
}
fine_tuned_validation_results.update({
    "false_positive_rate": float(fine_tuned_fpr),
    "threshold": 0.5,
    "number_of_validation_examples": int(
        fine_tuned_labels_all.size
    ),
})
FINE_TUNING_VALIDATION_METRICS_PATH.write_text(
    json.dumps(fine_tuned_validation_results, indent=2),
    encoding="utf-8",
)

baseline_results = json.loads(
    BASELINE_VALIDATION_METRICS_PATH.read_text(
        encoding="utf-8"
    )
)
comparison_metrics = [
    "binary_accuracy",
    "loss",
    "roc_auc",
    "pr_auc",
    "precision",
    "recall",
    "false_positive_rate",
]
baseline_comparison_df = pd.DataFrame({
    "frozen_baseline": [
        baseline_results[name]
        for name in comparison_metrics
    ],
    "fine_tuned": [
        fine_tuned_validation_results[name]
        for name in comparison_metrics
    ],
}, index=comparison_metrics)
baseline_comparison_df["difference"] = (
    baseline_comparison_df["fine_tuned"]
    - baseline_comparison_df["frozen_baseline"]
)
baseline_comparison_df.index.name = "metric"
baseline_comparison_df.to_csv(BASELINE_COMPARISON_PATH)

display(baseline_comparison_df.round(4))
print("Saved:", FINE_TUNING_VALIDATION_METRICS_PATH)
print("Saved:", BASELINE_COMPARISON_PATH)
print("Step 3B completed: fine-tuning evaluated.")


## Stop point after Step 3B

`Step 3B completed`が表示されたら、凍結baselineとfine-tuningモデルのValidation比較まで完了である。目標のBinary Accuracy約80%だけでなく、ROC-AUC、PR-AUC、Recall、FPRも確認する。

次のStep 4では、選択したモデルをTestingデータの既知ε=0.01, 0.1, 0.5と未知ε=0.05, 0.25, 1.0で個別評価する。
